## 설비 고장 예측 - Improved

**접근법 (copy4 기반)**
- 시간순 80/20 분리 → 훈련 구간 내 순수 정상 시퀀스만 사용
- `HIDDEN_SIZE=8`, 50 epochs
- Stage 2 분류: 전체 슬라이딩 윈도우 후 고장 유형 필터링 (`No Failure`, `Random Failures` 제외)

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from src.config import *
from src.models import LSTMAutoencoder, LSTMClassifier
from src.data_utils import load_and_engineer, prepare_autoencoder_data, prepare_classifier_data
from src.train_utils import train_autoencoder, compute_errors
from src.visualization import (
    plot_loss_curve, plot_error_distribution,
    plot_error_timeseries, plot_confusion_matrix
)

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')

### 1. 데이터 전처리

In [ ]:
df = load_and_engineer(DATA_PATH)

# 시간순 80/20 분리 (데이터 누수 방지)
split_idx = int(len(df) * TRAIN_RATIO)
train_raw = df.iloc[:split_idx].reset_index(drop=True)
test_raw = df.iloc[split_idx:].reset_index(drop=True)

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_raw[FEATURES])
test_scaled = scaler.transform(test_raw[FEATURES])

# 훈련: 윈도우 내 고장이 하나도 없는 순수 정상 시퀀스만 추출
X_train, y_train = prepare_autoencoder_data(
    train_scaled, train_raw['Target'].values, SEQUENCE_LENGTH, is_train=True
)
X_test, y_test = prepare_autoencoder_data(
    test_scaled, test_raw['Target'].values, SEQUENCE_LENGTH, is_train=False
)
y_test_true = test_raw['Target'].values[SEQUENCE_LENGTH - 1:]

print(f'훈련 (순수 정상) 시퀀스: {X_train.shape}')
print(f'테스트 시퀀스: {X_test.shape}')

In [ ]:
train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
train_eval_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### 2. 오토인코더 학습 (Stage 1)

In [ ]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

model = LSTMAutoencoder(INPUT_SIZE, IMPROVED_HIDDEN_SIZE, SEQUENCE_LENGTH).to(device)
history = train_autoencoder(
    model, train_loader, test_loader,
    IMPROVED_EPOCHS, LR, device,
    f'{CHECKPOINT_DIR}/autoencoder_v2.pth'
)

In [ ]:
plot_loss_curve(history)

### 3. 이상 탐지 평가 (Stage 1)

In [ ]:
model.load_state_dict(torch.load(f'{CHECKPOINT_DIR}/autoencoder_v2.pth'))

train_errors = compute_errors(model, train_eval_loader, device)
test_errors = compute_errors(model, test_loader, device)

# 퍼센타일별 임계값 비교
for p in [95, 90, 80]:
    threshold = np.percentile(train_errors, p)
    y_pred = (test_errors > threshold).astype(int)
    print(f'[{p}th percentile | threshold: {threshold:.6f}]')
    print(classification_report(y_test_true, y_pred, target_names=['Normal', 'Anomaly'], zero_division=0))

In [ ]:
threshold = np.percentile(train_errors, 95)
y_pred = (test_errors > threshold).astype(int)

plot_error_distribution(train_errors, test_errors, y_test_true, threshold)
plot_error_timeseries(test_errors, y_test_true, threshold,
                      save_path=f'{OUTPUT_DIR}/error_timeseries.png')
plot_confusion_matrix(y_test_true, y_pred, ['Normal', 'Anomaly'],
                      '이상 탐지 혼동 행렬 (Stage 1)',
                      save_path=f'{OUTPUT_DIR}/confusion_matrix_stage1.png')

### 4. 고장 유형 분류 (Stage 2)

`No Failure`와 `Random Failures`를 제외한 4개 고장 유형만 분류한다.

In [ ]:
all_scaled = scaler.transform(df[FEATURES])
X_fail, y_fail, le = prepare_classifier_data(
    all_scaled, df['Failure Type'].values, SEQUENCE_LENGTH, EXCLUDE_FAILURE_TYPES
)
print('클래스:', le.classes_)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_fail, y_fail, test_size=0.2, random_state=42, stratify=y_fail
)

num_classes = len(le.classes_)
clf_train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)), batch_size=16, shuffle=True
)
clf_test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_te), torch.LongTensor(y_te)), batch_size=16, shuffle=False
)

classifier = LSTMClassifier(INPUT_SIZE, 32, num_classes).to(device)
clf_criterion = torch.nn.CrossEntropyLoss()
clf_optimizer = torch.optim.Adam(classifier.parameters(), lr=LR)

CLF_EPOCHS = 50
for epoch in range(1, CLF_EPOCHS + 1):
    classifier.train()
    for inputs, labels in clf_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        clf_optimizer.zero_grad()
        clf_criterion(classifier(inputs), labels).backward()
        clf_optimizer.step()

classifier.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in clf_test_loader:
        _, predicted = torch.max(classifier(inputs.to(device)), 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_, zero_division=0))
plot_confusion_matrix(all_labels, all_preds, le.classes_,
                      '고장 유형 분류 혼동 행렬 (Stage 2)',
                      save_path=f'{OUTPUT_DIR}/confusion_matrix_stage2.png')